In [ ]:
import os
import re
import json
import time
import html
import zipfile
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import TextVectorization

# =============================================================================
# 1. Local LLM Setup (Ollama)
# =============================================================================
# Make sure Ollama is running: `ollama serve`
# And you've pulled a model: `ollama pull llama3.2:3b`
OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
OLLAMA_MODEL = "llama3.2:3b"   # swap to "mistral:7b" for better quality

def call_local_llm(prompt, system_instruction=None):
    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": system_instruction})
    messages.append({"role": "user", "content": prompt})

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": messages,
            "temperature": 0.0,   # deterministic output, better for JSON
            "stream": False
        }
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

# =============================================================================
# 2. Load Deep Learning Assets
# =============================================================================
MODEL_PATH = os.path.join('..', 'models', 'bidi_lstm_balanced_model.keras')
VOCAB_PATH = os.path.join('..', 'models', 'vectorizer_vocabulary.json')
TEST_DATA_PATH = os.path.join('..', 'dataset', 'archive.zip')

print("Loading Bi-LSTM model and vocabulary...")
model = load_model(MODEL_PATH)
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    saved_vocabulary = json.load(f)

# Reconstruct training text vectorizer
inference_vectorizer = TextVectorization(
    max_tokens=10000,
    output_mode='int',
    output_sequence_length=150,
    vocabulary=saved_vocabulary
)

# =============================================================================
# 3. Load Test Dataset Subset
# =============================================================================
print("Loading test data from zip archive...")
with zipfile.ZipFile(TEST_DATA_PATH, 'r') as z:
    with z.open('drugsComTest_raw.csv') as f:
        df_test = pd.read_csv(f, sep=',')

# Take a clean slice of 100 rows
df_eval = df_test.head(100).copy()

# Guard against NaN/empty reviews
df_eval['review'] = df_eval['review'].fillna('').astype(str)

print(f"Successfully loaded {len(df_eval)} test rows for local LLM processing!")

# Clean text for Bi-LSTM prediction
def clean_text(text):
    text = str(text).lower()
    text = html.unescape(text)
    return text.strip()

print("Generating Bi-LSTM ratings...")
cleaned_reviews = df_eval['review'].apply(clean_text)
vectorized_inputs = inference_vectorizer(cleaned_reviews.values).numpy()
raw_preds = model.predict(vectorized_inputs, verbose=0)
df_eval['predicted_rating'] = np.argmax(raw_preds, axis=1) + 1

# =============================================================================
# 4. Prompt Engineering Alternatives (unchanged)
# =============================================================================

# --- ALTERNATIVE A: ZERO-SHOT STRUCTURED PROMPT ---
def get_prompt_alt_a(review):
    return f"""
You are analyzing a patient medical review. Your task is to extract an ultra-short summary and a binary sentiment.

Review to analyze: "{review}"

Return your response exactly as a valid JSON object with these two keys:
- "summary": A summary of the review that is strictly a MAXIMUM of 10 words.
- "sentiment": Output exactly either "positive" or "negative".
"""

# --- ALTERNATIVE B: FEW-SHOT SYSTEM-TAGGED PROMPT ---
system_instruction_alt_b = """
You are a specialized clinical data extraction assistant. Your job is to parse unstructured patient drug reviews into rigid, concise formats.
You must strictly follow these constraints:
1. The summary field must be 10 words or fewer.
2. The sentiment field must be exactly 'positive' or 'negative'.
3. Respond ONLY with a valid JSON block containing the keys "summary" and "sentiment". Do not include markdown code block formatting or backticks.
"""

def get_prompt_alt_b(review):
    return f"""
<examples>
Review: "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
Output: {{"summary": "Life-changing drug eliminated all chronic symptoms quickly with no side-effects.", "sentiment": "positive"}}

Review: "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
Output: {{"summary": "Severe side effects including intense nausea and headaches immediately.", "sentiment": "negative"}}
</examples>

<target_review>
{review}
</target_review>

Output JSON:
"""

# =============================================================================
# 5. Execution Loop (no rate limits, no sleep needed)
# =============================================================================
def run_genai_pipeline(prompt_style="A"):
    summaries = []
    sentiments = []

    print(f"\nRunning Generative Track: Alternative {prompt_style}...")

    for idx, (current_idx, row) in enumerate(df_eval.iterrows()):
        review_text = row['review']
        print(f"Processing row {idx + 1}/{len(df_eval)}...", end="\r")

        max_retries = 3
        success = False

        for attempt in range(max_retries):
            try:
                if prompt_style == "A":
                    raw_text = call_local_llm(get_prompt_alt_a(review_text))
                else:
                    raw_text = call_local_llm(
                        get_prompt_alt_b(review_text),
                        system_instruction=system_instruction_alt_b
                    )

                # Strip markdown code fences if the model wraps output in ```json ... ```
                # Small local models sometimes do this despite being told not to
                raw_text = re.sub(r'```json|```', '', raw_text).strip()

                # Parse JSON output
                data = json.loads(raw_text)
                summaries.append(data.get("summary", "Error parsing summary"))
                sentiments.append(data.get("sentiment", "Error parsing sentiment"))
                success = True
                break

            except json.JSONDecodeError:
                # Local models can occasionally produce malformed JSON — just retry
                print(f"\nJSON parse failed on row {idx + 1}, retrying... "
                      f"(Attempt {attempt + 1}/{max_retries})")

            except Exception as e:
                # Full error log — not truncated
                print(f"\nError on row {idx + 1}: {type(e).__name__}: {e}")
                break

        if not success:
            summaries.append("LLM Execution Error")
            sentiments.append("error")

        # No sleep needed — local LLM has no rate limit

    df_output = df_eval[['drugName', 'condition', 'review', 'rating', 'predicted_rating']].copy()
    df_output['generated_summary'] = summaries
    df_output['identified_sentiment'] = sentiments
    return df_output

# Run both strategies sequentially
output_alt_a = run_genai_pipeline(prompt_style="A")
output_alt_b = run_genai_pipeline(prompt_style="B")

# =============================================================================
# 6. Export to Excel
# =============================================================================
output_dir = os.path.join('..', 'outputs')
os.makedirs(output_dir, exist_ok=True)

output_alt_a.to_excel(os.path.join(output_dir, 'alternative_a_zero_shot.xlsx'), index=False)
output_alt_b.to_excel(os.path.join(output_dir, 'alternative_b_few_shot.xlsx'), index=False)

print("\nBoth Excel files exported to the outputs/ directory.")

2026-06-13 16:14:54.579684: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading Bi-LSTM model and vocabulary...
Loading test data from zip archive...
Successfully loaded 100 test rows for local LLM processing!
Generating Bi-LSTM ratings...

Running Generative Track: Alternative A...
Processing row 100/100...
Running Generative Track: Alternative B...
Processing row 100/100...
Both Excel files exported to the outputs/ directory.
